# Itinera · Fine-tuning LoRA en Colab

La idea de este notebook es hacer un fine-tuning sencillo sobre un modelo abierto pequeño. No hace falta que el modelo aprenda información de viajes actualizada, para eso la aplicación usa búsqueda web y otras fuentes. Lo que quiero hacer es comprobar si podemos darle al modelo una forma concreta de organizar los itinerarios, alineada con el tono de la empresa (en este caso ficticia).

De todos modos, para la web, voy a acabar usando la API de OpenAI, ya que crear un servidor para exponer la API del modelo finetuneado e incorporarlo a la aplicación es algo más complejo. Estaba intentando alojar el nuevo modelo en hugging face, pero crear un servidor dinámico requería una cuenta de pago.



In [8]:
!pip -q uninstall -y torchao
!pip -q install -U transformers datasets peft trl accelerate huggingface_hub


## 1. Dataset

Vamos a crear un dataset sencillo que tenga varios ejemplos sobre posibles ciudades y perfiles de viajeros. Cada ejemplo contiene una preferencia distinta y una respuesta objetivo. La idea no es que el modelo aprenda esos viajes concretos, sino que aprenda reglas de organización: separar presupuestos, agrupar por zonas, respetar el ritmo y no inventar precios u horarios.


In [9]:
from datasets import Dataset
from itertools import product

cities = {
    'Lisboa': ['Alfama y Baixa', 'Belém', 'Chiado y Bairro Alto'],
    'Sevilla': ['Santa Cruz y centro', 'Triana', 'Alameda y Macarena'],
    'Roma': ['Centro Storico', 'Trastevere', 'Villa Borghese'],
    'París': ['Marais y centro', 'Montmartre', 'Saint-Germain'],
    'Kioto': ['Higashiyama', 'Arashiyama', 'Gion y centro'],
    'Oporto': ['Ribeira', 'Cedofeita', 'Foz y Duero'],
    'Granada': ['Albaicín y centro', 'Realejo', 'Sacromonte'],
    'Ámsterdam': ['Centro y Jordaan', 'Museumplein', 'De Pijp'],
}
profiles = [
    ('cultura y gastronomía', 75, 'equilibrado', 'quiero ver lo importante sin ir corriendo'),
    ('historia y vida local', 50, 'ajustado', 'prefiero caminar y gastar poco'),
    ('naturaleza y fotografía', 95, 'cómodo', 'me interesa tener tiempo para parar'),
    ('arte y paseos', 65, 'tranquilo', 'no quiero más de tres paradas al día'),
    ('comida y barrios', 80, 'relajado', 'quiero dejar hueco para improvisar'),
]
durations = [2, 3, 4]
openings = ['Organiza', 'Prepárame', 'Diseña', 'Ayúdame a planificar', 'Quiero una propuesta para']

def answer(city, zones, interests, budget, pace, extra, days):
    lines = [
        f'PROPUESTA · {city} · {days} días',
        f'Enfoque: {interests}. Ritmo: {pace}.',
        'Viaje y alojamiento: compáralos aparte según las fechas y busca una zona bien conectada.',
        f'Gasto local objetivo: alrededor de {budget} € por persona y día para actividades, comida y transporte.',
    ]
    for day in range(days):
        zone = zones[day % len(zones)]
        lines += [
            f'Día {day + 1} · {zone}',
            f'· Mañana: actividad principal en {zone}, sin encadenar visitas innecesarias.',
            '· Mediodía: comida local o mercado cercano.',
            '· Tarde: paseo, museo o plan flexible en el mismo barrio.',
            '· Noche: cena tranquila o ambiente local según energía.',
        ]
    lines += [
        f'Nota de organización: {extra}.',
        'Aviso: no doy por confirmados precios, horarios ni disponibilidad. Revísalos en fuentes oficiales antes de reservar.'
    ]
    return '\n'.join(lines)

rows = []
for i, ((city, zones), (interests, budget, pace, extra), days) in enumerate(product(cities.items(), profiles, durations)):
    prompt = f'{openings[i % len(openings)]} {days} días en {city}. Me interesan {interests}, tengo unos {budget} € al día y un ritmo {pace}. {extra}.'
    rows.append({'messages': [
        {'role': 'system', 'content': 'Eres Itinera. Organizas viajes de forma clara, prudente y realista. Nunca inventes precios, horarios, disponibilidad ni sitios concretos si no se te han proporcionado.'},
        {'role': 'user', 'content': prompt},
        {'role': 'assistant', 'content': answer(city, zones, interests, budget, pace, extra, days)},
    ]})

dataset = Dataset.from_list(rows).train_test_split(test_size=0.15, seed=42)
print(dataset)
print('\nEjemplo de entrenamiento:\n', dataset['train'][0]['messages'][-1]['content'])


DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 102
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 18
    })
})

Ejemplo de entrenamiento:
 PROPUESTA · Oporto · 4 días
Enfoque: comida y barrios. Ritmo: relajado.
Viaje y alojamiento: compáralos aparte según las fechas y busca una zona bien conectada.
Gasto local objetivo: alrededor de 80 € por persona y día para actividades, comida y transporte.
Día 1 · Ribeira
· Mañana: actividad principal en Ribeira, sin encadenar visitas innecesarias.
· Mediodía: comida local o mercado cercano.
· Tarde: paseo, museo o plan flexible en el mismo barrio.
· Noche: cena tranquila o ambiente local según energía.
Día 2 · Cedofeita
· Mañana: actividad principal en Cedofeita, sin encadenar visitas innecesarias.
· Mediodía: comida local o mercado cercano.
· Tarde: paseo, museo o plan flexible en el mismo barrio.
· Noche: cena tranquila o ambiente local según energía.
Día 3 · Foz y Duero
· Mañan

## 2. Modelo base

Uso Qwen3-0.6B porque cabe en una GPU gratuita de Colab. Vamos a guardar una respuesta con la misma petición que usaremos al final para comparar el resultado.

In [10]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'Qwen/Qwen3-0.6B'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float16, device_map='auto')

test_messages = [
    {'role': 'system', 'content': 'Eres Itinera. Organizas viajes de forma clara, prudente y realista. Nunca inventes precios, horarios, disponibilidad ni sitios concretos si no se te han proporcionado.'},
    {'role': 'user', 'content': 'Organiza 3 días en Lisboa. Me interesan cultura y gastronomía, tengo 70 € al día y quiero un ritmo tranquilo. No quiero ir con prisa y prefiero caminar.'},
]

def generate(messages):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=420, do_sample=False, repetition_penalty=1.08)
    return tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

before = generate(test_messages)
print('\nANTES DEL FINE_TUNING\n', before)


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]


ANTES DEL FINE_TUNING
 ¡Gracias por tu interés! Organizar un viaje de 3 días en **Lisboa** con tiendas de cultura, gastronomía y un ritmo tranquilo es una excelente idea. Aquí tienes una propuesta detallada:

---

### **Día 1: Lugar para la cultura y el arte**

- **Origen**: **Catedral de Lisboa**  
  - Descubre la historia y la arquitectura del lugar más famoso de la capital.  
  - Visita el **Palácio de la Moneda**, donde hay una exposición sobre la historia de la moneda.

- **Actividades**:  
  - Caminar por las calles sin perder el equilibrio.  
  - Comer en restaurantes típicos como **La Candelabro** o **El Poblet**.  
  - Participa en un taller de pintura o arte en la plaza.

- **Costo**: 70 €/día → 210 € total.

---

### **Día 2: Elaboración de la gastronomía**

- **Origen**: **Plaza de los Olivos**  
  - Comer en lugares como **El Tamarindo** o **El Poblet**.  
  - Verificar si hay opciones de comida regional en el lugar.

- **Actividades**:  
  - Caminar por las calles y disf

## 3. Ajuste LoRA

El fine-tuning con LoRA (Low-Rank Adaptation) se basa en congelar los pesos del modelo, y construir unas nuevas matrices de pesos de menor dimensión que sí se van a entrenar, y que se sumarán a las proyecciones de atención. Es como una especie de capa que se construye encima del modelo base, por lo que se pueden construir varias capas LoRA encima del mismo modelo para diferentes fines.


In [11]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    task_type='CAUSAL_LM',
)
training_args = SFTConfig(
    output_dir='/content/itinera-lora',
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=1.5e-4,
    logging_steps=5,
    eval_strategy='epoch',
    save_strategy='epoch',
    max_length=768,
    fp16=True,
    report_to='none',
    seed=42,
)
trainer = SFTTrainer(
    model=model, args=training_args,
    train_dataset=dataset['train'], eval_dataset=dataset['test'],
    processing_class=tokenizer, peft_config=peft_config,
)
trainer.train()
trainer.save_model('/content/itinera-lora/final')


Tokenizing train dataset:   0%|          | 0/102 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/102 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/102 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/102 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/18 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/18 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/18 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/18 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,0.351500,0.237765,0.396318,47275.000000,0.955660
2,0.089249,0.086943,0.136465,94550.000000,0.983769
3,0.057349,0.054112,0.089316,141825.000000,0.989236
4,0.045490,0.048410,0.072384,189100.000000,0.989076
5,0.044027,0.045513,0.069164,236375.000000,0.989326


## 4. Comparación

Comparamos los resultados del modelo base con los resultados conseguidos con el fine-tuning con LoRA. La comparación usa la misma petición antes y después. Vamos a comprobar si el nuevo modelo sigue mejor las reglas que hemos definido para Itinera.

Por eso miramos dos cosas: si aparecen los bloques importantes de la propuesta y si el modelo evita confirmar datos que no tiene. Uno de los problemas del modelo base es que inventa datos que no son reales (como la Plaza de los Olivos en Lisboa).

In [12]:
metrics = trainer.evaluate()
after = generate(test_messages)

checks = ['PROPUESTA', 'Viaje y alojamiento', 'Gasto local', 'Día 1', 'Día 2', 'Día 3', 'Aviso']
def check_rules(text):
    return {rule: rule.lower() in text.lower() for rule in checks}

before_rules = check_rules(before)
after_rules = check_rules(after)
print('Pérdida de validación:', round(metrics['eval_loss'], 4))
print('Reglas de estructura antes:', sum(before_rules.values()), '/ 7')
print('Reglas de estructura después:', sum(after_rules.values()), '/ 7')


Training Loss,Validation Loss,Epoch,Entropy,Num Tokens,Mean Token Accuracy
0.044027,0.045513,5,0.069164,236375.000000,0.989326


Pérdida de validación: 0.0455
Reglas de estructura antes: 4 / 7
Reglas de estructura después: 7 / 7


In [13]:
from html import escape
from IPython.display import HTML, display
import pandas as pd

comparison = pd.DataFrame({
    'Regla que queremos enseñar al modelo': list(before_rules.keys()),
    'Qwen base': ['✓' if value else '—' for value in before_rules.values()],
    'Qwen + Itinera LoRA': ['✓' if value else '—' for value in after_rules.values()],
})
display(comparison)

display(HTML(f"""
<div style="display:flex; gap:18px; align-items:flex-start; font-family:Arial, sans-serif;">
  <div style="width:50%; border:1px solid #ddd; border-radius:10px; padding:14px;">
    <h3 style="margin-top:0;">Antes · Qwen base</h3>
    <div style="white-space:pre-wrap; max-height:460px; overflow-y:auto; line-height:1.45;">{escape(before)}</div>
  </div>
  <div style="width:50%; border:1px solid #5b8c5a; border-radius:10px; padding:14px; background:#f7fcf6;">
    <h3 style="margin-top:0;">Después · Qwen + Itinera LoRA</h3>
    <div style="white-space:pre-wrap; max-height:460px; overflow-y:auto; line-height:1.45;">{escape(after)}</div>
  </div>
</div>
"""))

print(f"\nResultado: el modelo base cumple {sum(before_rules.values())}/7 reglas y el modelo con LoRA {sum(after_rules.values())}/7.")


,Regla que queremos enseñar al modelo,Qwen base,Qwen + Itinera LoRA
0,PROPUESTA,✓,✓
1,Viaje y alojamiento,—,✓
2,Gasto local,—,✓
3,Día 1,✓,✓
4,Día 2,✓,✓
5,Día 3,✓,✓
6,Aviso,—,✓



Resultado: el modelo base cumple 4/7 reglas y el modelo con LoRA 7/7.


## Conclusión

Aunque parezca que no haya mejora, LoRA ha conseguido que el modelo pequeño siga mejor el formato y las reglas de Itinera. Sin embargo, como los ejemplos de entrenamiento son bastante estructurados, la respuesta final queda también más rígida y menos natural que la del modelo base.

No creo que tenga sentido vender esto como que Qwen+LoRA ahora organiza viajes mejor que GPT, que es el que estamos usando en la web. La mejora real está en que se comporta de una forma más controlada que el Qwen base: separa presupuestos, organiza por días y evita dar datos erróneos como si fueran reales. Para generar un modelo que realmente fuera útil haría falta un dataset mucho más grande y refinado.

Vamos a dejar un modelo GPT en la aplicación, que además tiene recuperación de información web mediante APIs porque es la opción que da una experiencia más útil para alguien que de verdad va a preparar un viaje.

Integrar este modelo habría requerido alojar tanto Qwen como el adaptador en un servicio de inferencia (lo he intentado en HF, pero necesitaba cuenta de pago), y creo que realmente no merece la pena.